In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pathlib
import matplotlib.pyplot as plt
from sklearn import linear_model

In [ ]:
def prob_to_odds(x):
    return np.log(x / (1 - x - 1e-6))


def odds_to_prob(x):
    return np.exp(x) / (1 + np.exp(x))


def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Stabilizing to prevent overflow
    return exp_x / np.sum(exp_x)

In [ ]:
def make_sparse(adata, percentage: float, clf, rng):
    logits = clf.predict(np.log(adata.X.data.reshape(-1, 1) + 1))
    idx = rng.choice(
        range(len(adata.X.data)),
        size=int(percentage * len(adata.X.data)),
        p=softmax(logits),
        replace=False,
    )
    adata.X.data[idx] = 0.0
    adata.X.eliminate_zeros()
    return adata


def make_sparse_unif(adata, percentage: float, rng):
    idx = rng.choice(
        range(len(adata.X.data)),
        size=int(percentage * len(adata.X.data)),
        replace=False,
    )
    adata.X.data[idx] = 0.0
    adata.X.eliminate_zeros()
    return adata

In [ ]:
rng = np.random.default_rng(0)

In [ ]:
marker_genes = pd.read_csv("annotations/neftel_6.csv")

In [ ]:
adata = sc.read_h5ad(
    "results/signatures/gbm/data/preprocessed/neftel_smartseq2_brain.h5ad"
)
print(adata.shape)
adata.X = adata.layers["counts"].copy()
sc.pp.filter_genes(adata, min_cells=10)
# sc.pp.filter_genes(adata, min_counts=1000)
mean = adata.X.mean(0).A[0]
n_zero = 1 - (adata.X != 0).mean(0).A[0]
print(adata.shape)

In [ ]:
adata = sc.read_h5ad(
    "results/signatures/sparse/data/preprocessed/neftel_0.25_2_all_0.h5ad"
)

In [ ]:
len(adata.X.nonzero()[0]) / (adata.n_obs * adata.n_vars)

In [ ]:
clf = linear_model.LinearRegression()
clf.fit(np.log(mean.reshape((-1, 1)) + 1), prob_to_odds(n_zero))
line_scale = np.linspace(0.0, 10)
fitted = odds_to_prob(clf.predict(line_scale.reshape(-1, 1)))

In [ ]:
color = [
    "red" if gene else "blue"
    for gene in adata.var_names.isin(marker_genes.values.flatten())
]
size = [
    0.5 if gene else 0.01
    for gene in adata.var_names.isin(marker_genes.values.flatten())
]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(np.log(mean + 1), 100 * n_zero, s=size, c=color)
ax.scatter([], [], c="red", s=50, label="Marker gene")
ax.scatter([], [], c="blue", s=50, label="Non-marker gene")

ax.plot(line_scale, 100 * fitted, color="black", label="Fitted model")
ax.set_ylabel("Percentage of zero elemetns", fontsize=16)
ax.set_xlabel("Log mean expression", fontsize=16)

ax.legend()

fig.savefig("figures/plots/sim.svg")

In [ ]:
from itertools import product

for random_seed, percentage in product(range(5), [0.25, 0.5, 0.75, 0.9]):
    rng = np.random.default_rng(random_seed)
    bdata = adata.copy()
    bdata = make_sparse_unif(bdata, percentage=percentage, rng=rng)
    bdata.layers["counts"] = bdata.X.copy()
    sc.pp.filter_genes(bdata, min_cells=2)
    sc.pp.filter_cells(bdata, min_genes=5)
    sc.pp.normalize_total(bdata, target_sum=1e5)
    sc.pp.log1p(bdata, base=2)
    bdata.write_h5ad(f"data/sparse/neftel_{percentage}_{random_seed}_unif.h5ad")

In [ ]:
from itertools import product

for random_seed, percentage in product(range(5), [0.25, 0.5, 0.75, 0.9]):
    rng = np.random.default_rng(random_seed)
    bdata = adata.copy()
    bdata = make_sparse(bdata, percentage=percentage, clf=clf, rng=rng)
    bdata.layers["counts"] = bdata.X.copy()
    sc.pp.filter_genes(bdata, min_cells=2)
    sc.pp.filter_cells(bdata, min_genes=5)
    sc.pp.normalize_total(bdata, target_sum=1e5)
    sc.pp.log1p(bdata, base=2)
    bdata.write_h5ad(f"data/sparse/neftel_{percentage}_{random_seed}.h5ad")

In [ ]:
for data_path in pathlib.Path("data/sparse").iterdir():
    print(f"{data_path.stem}:")
    print(f"\tdata_path: {data_path}")
    print("\tmem_scaling: 0.5")
    print("\tsignatures:")
    print("\t\t- neftel_4")